# Data Extraction & Web Scraping — Self-Study Guide

Welcome! This notebook teaches you how to collect data from the internet using three essential techniques:

1. **REST API** — request structured data from a server using standard HTTP calls
2. **GraphQL** — ask for exactly the fields you need, nothing more
3. **Web Scraping** — extract data directly from HTML when no API exists

**By the end of this notebook you will be able to:**
- Authenticate and paginate through a real REST API (GitHub)
- Write GraphQL queries to fetch precise data
- Parse HTML with BeautifulSoup and handle multi-page scrapes

> **Who this is for:** You know basic Python (variables, loops, functions). You've never called an API or scraped a website before. No worries — everything is explained step by step.

## What You'll Build

By the end of this notebook you'll have extracted five real datasets:

| # | Dataset | How | Output |
|---|---------|-----|--------|
| 1 | `pandas` library release history | REST API | `pandas_releases` DataFrame + `pandas_releases` DuckDB table |
| 2 | Top contributors by commit count | REST API | `contributors` DataFrame + `pandas_contributors` DuckDB table |
| 3 | GraphQL query of `pandas` releases | GraphQL | `graphql_releases` DataFrame |
| 4 | Countries of the world (name, capital, population, area) | Web scraping | `countries_df` DataFrame |
| 5 | NHL hockey team stats (all 24 pages) | Web scraping + pagination | `hockey_stats` DataFrame |

Here's a sneak peek at what `pandas_releases` will look like:

```
   version          published_at                          summary
0  v2.2.1   2024-03-18 15:30:00+00:00  ## What's new in 2.2.1...
1  v2.2.0   2024-01-19 17:00:00+00:00  ## What's new in 2.2.0...
2  v2.1.4   2024-01-08 20:00:00+00:00  ## What's new in 2.1.4...
...
```

---
## Setup

Before we write any code, let's get our tools ready.

### Libraries we'll use

| Library | What it does |
|---------|-------------|
| `requests` | Makes HTTP calls to APIs and websites |
| `python-dotenv` | Loads secret keys from a `.env` file so they stay out of your code |
| `pandas` | Creates DataFrames — like spreadsheets in Python |
| `sqlalchemy` | Lets pandas talk to databases |
| `duckdb` | A fast, file-based database — we'll save our results here |
| `beautifulsoup4` | Parses HTML so we can extract data from web pages |

Install them if needed:
```bash
pip install requests python-dotenv pandas sqlalchemy duckdb duckdb-engine beautifulsoup4
```

### Step 1 of 2: Get a GitHub Personal Access Token

We need a token so GitHub knows who we are and lets us call their API.

**Follow these steps:**

1. Go to [github.com](https://github.com) and sign in
2. Click your profile photo (top-right) → **Settings**
3. Scroll down the left sidebar → click **Developer settings** (at the very bottom)
4. Click **Personal access tokens** → **Fine-grained tokens** → **Generate new token**
5. Give it a name (e.g. `self-study-scraping`)
6. Set **Expiration** to 30 days
7. Under **Repository access**, select **Public repositories (read-only)**
8. Scroll to the bottom → click **Generate token**
9. **Copy the token immediately** — GitHub only shows it once!

> ⚠️ Keep your token private. Never paste it directly into a notebook or commit it to GitHub.

In [ ]:
import os

# Run this cell ONCE to create your .env file.
# After running, open the .env file and paste your token in place of the placeholder.

env_path = '../.env'

if os.path.exists(env_path):
    print(f"ℹ️  .env file already exists at '{env_path}'.")
    print("   Open it and make sure GITHUB_TOKEN is set to your actual token.")
else:
    with open(env_path, 'w') as f:
        f.write("# GitHub personal access token\n")
        f.write("GITHUB_TOKEN='<PASTE-YOUR-TOKEN-HERE>'\n")
    print(f"✅ Created '{env_path}'")
    print("   Now open the file and replace <PASTE-YOUR-TOKEN-HERE> with your actual token.")

### Step 2 of 2: Add your token to the .env file

1. Find the `.env` file in the **root folder** of this project (one level above `notebooks/`)
2. Open it in a text editor
3. Replace `<PASTE-YOUR-TOKEN-HERE>` with your actual token, like this:

```
GITHUB_TOKEN='ghp_xxxxxxxxxxxxxxxxxxxx'
```

4. Save and close the file

> 📁 The `.env` file is listed in `.gitignore` so it will **never** be pushed to GitHub.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # 👈 reads the .env file and loads GITHUB_TOKEN into the environment
access_token = os.getenv('GITHUB_TOKEN')

# Verify the token loaded correctly (prints first/last 4 chars only — never print the full token!)
if access_token and access_token != '<PASTE-YOUR-TOKEN-HERE>':
    if len(access_token) >= 8:
        print(f"✅ Token loaded: {access_token[:4]}...{access_token[-4:]}")
    else:
        print("⚠️  Token looks too short — double-check you copied it in full.")
else:
    print("❌ Token not found or still a placeholder. Check your .env file.")

---
## Chapter 1 — REST API

### The Mission

You've joined a team that tracks the health of popular open-source libraries. Your first task: **collect the full release history of the `pandas` library** — every version, when it was published, and what changed.

Pandas hosts all of this on GitHub. GitHub exposes it via a **REST API** — a web service that returns structured data when you send it the right request. Let's learn how.

### Concept: How a REST API Works

When you call an API, your Python script sends an **HTTP request** to a server. The server processes it and sends back an **HTTP response** containing data (usually in JSON format).

<svg viewBox="0 0 620 110" xmlns="http://www.w3.org/2000/svg" style="font-family:sans-serif;font-size:12px;max-width:620px;display:block;margin:16px 0;">
  <rect x="10" y="35" width="120" height="42" rx="6" fill="#dbeafe" stroke="#3b82f6" stroke-width="1.5"/>
  <text x="70" y="57" text-anchor="middle" fill="#1e40af" font-weight="bold">Your Script</text>
  <text x="70" y="71" text-anchor="middle" fill="#1e40af" font-size="10">(requests.get)</text>
  <rect x="490" y="35" width="120" height="42" rx="6" fill="#dcfce7" stroke="#22c55e" stroke-width="1.5"/>
  <text x="550" y="57" text-anchor="middle" fill="#166534" font-weight="bold">GitHub API</text>
  <text x="550" y="71" text-anchor="middle" fill="#166534" font-size="10">api.github.com</text>
  <defs>
    <marker id="arr-blue" markerWidth="8" markerHeight="6" refX="7" refY="3" orient="auto"><polygon points="0 0,8 3,0 6" fill="#3b82f6"/></marker>
    <marker id="arr-green" markerWidth="8" markerHeight="6" refX="7" refY="3" orient="auto"><polygon points="0 0,8 3,0 6" fill="#22c55e"/></marker>
  </defs>
  <line x1="130" y1="50" x2="488" y2="50" stroke="#3b82f6" stroke-width="1.5" marker-end="url(#arr-blue)"/>
  <text x="310" y="44" text-anchor="middle" fill="#1d4ed8" font-size="11">① GET /repos/pandas-dev/pandas/releases   +   Authorization header</text>
  <line x1="490" y1="63" x2="132" y2="63" stroke="#22c55e" stroke-width="1.5" marker-end="url(#arr-green)"/>
  <text x="310" y="80" text-anchor="middle" fill="#15803d" font-size="11">② 200 OK   +   JSON array of release objects</text>
</svg>

**Key terms:**

| Term | What it means |
|------|---------------|
| **Endpoint** | The URL you call, e.g. `https://api.github.com/repos/pandas-dev/pandas/releases` |
| **HTTP verb** | The type of action — `GET` = read data, `POST` = send data |
| **Header** | Extra info sent with the request — e.g. your identity token |
| **Status code** | A 3-digit number in the response: `200` = success, `4xx` = your error, `5xx` = server error |
| **JSON** | The data format APIs return — looks like a Python dict/list |
| **Bearer token** | Your API key, passed in the `Authorization` header |

### Concept: JSON is just Python dicts and lists

JSON responses from APIs look like this:

```json
[
  {
    "tag_name": "v2.2.1",
    "published_at": "2024-03-18T15:30:00Z",
    "body": "## What's new in 2.2.1 ..."
  },
  {
    "tag_name": "v2.2.0",
    "published_at": "2024-01-19T17:00:00Z",
    "body": "## What's new in 2.2.0 ..."
  }
]
```

- The outer `[...]` means it's a **list**
- Each `{...}` is a **dict** with key-value pairs
- In Python: `response.json()` gives you the list. `response.json()[0]["tag_name"]` gives you `"v2.2.1"`

### Worked Example: Fetching pandas Releases

Let's call the GitHub releases endpoint step by step. We'll explain every line.

In [ ]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()
access_token = os.getenv('GITHUB_TOKEN')

response = requests.get(
    "https://api.github.com/repos/pandas-dev/pandas/releases",  # 👈 the endpoint
    headers={
        "Accept": "application/vnd.github+json",   # 👈 tells GitHub we want JSON format
        "Authorization": f"Bearer {access_token}", # 👈 proves who we are
        "X-GitHub-Api-Version": "2022-11-28"       # 👈 pins to a stable API version
    }
)

In [ ]:
# A status code of 200 means success.
# Print it so we can confirm the request worked before going further.
print(response.status_code)

> **Troubleshooting — if you don't see `200`:**
>
> | Status code | Meaning | Fix |
> |-------------|---------|-----|
> | `401` | Unauthorized — token missing or wrong | Check your `.env` file. Print `access_token` to see if it loaded. |
> | `403` | Forbidden — token doesn't have permission | Create a new token with "Public repositories (read-only)" access |
> | `404` | Not found — URL is wrong | Double-check the endpoint URL for typos |
> | `422` | Unprocessable — bad query parameter | Check `per_page` / `page` values |
>
> **Quick debug:** Run `print(access_token)` to see if the token loaded. If it shows `None`, your `.env` file isn't being found.

In [ ]:
# .json() converts the response body from raw text into a Python list/dict
releases = response.json()
print(type(releases))  # should be <class 'list'>
print(len(releases))   # default page size is 30

In [ ]:
# Look at the first release — you'll see a lot of fields we don't need!
# We'll learn how to ask for only what we want in Chapter 2 (GraphQL).
releases[0]

In [ ]:
# We care about three fields: tag_name, published_at, body
print("Version:", releases[0]['tag_name'])
print("Published:", releases[0]['published_at'])

### Pagination — Getting More Than 30 Results

By default GitHub returns 30 results per page. `pandas` has 100+ releases, so we need to page through.

Two query parameters control this:
- `per_page=100` — return up to 100 results per page (the maximum)
- `page=2` — get the second page of results

We add them to the URL after a `?`, separated by `&`:
```
/releases?per_page=100&page=1
/releases?per_page=100&page=2
```

In [ ]:
# Get page 1 with 100 results per page
response_p1 = requests.get(
    "https://api.github.com/repos/pandas-dev/pandas/releases?per_page=100&page=1",
    headers={
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {access_token}",
        "X-GitHub-Api-Version": "2022-11-28"
    }
)
releases_p1 = response_p1.json()
print(f"Page 1: {len(releases_p1)} releases")
print("Oldest on page 1:", releases_p1[-1]['tag_name'])

In [ ]:
# Get page 2 (releases older than what page 1 returned)
# We fetch it here to demonstrate pagination — we'll use page 1's 100 entries for our DataFrame
response_p2 = requests.get(
    "https://api.github.com/repos/pandas-dev/pandas/releases?per_page=100&page=2",
    headers={
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {access_token}",
        "X-GitHub-Api-Version": "2022-11-28"
    }
)
releases_p2 = response_p2.json()
print(f"Page 2: {len(releases_p2)} releases")
if releases_p2:
    print("Oldest on page 2:", releases_p2[-1]['tag_name'])
else:
    print("Page 2 is empty — all releases fit on page 1")

### Building a DataFrame

We only need three fields from each release. Let's use a list comprehension to extract them, then build a DataFrame.

In [ ]:
import pandas as pd

# Extract only the fields we need from page 1 (the most recent 100 releases)
records = [
    {
        "version": r["tag_name"],          # 👈 the version number e.g. "v2.2.1"
        "published_at": r["published_at"], # 👈 ISO 8601 date string
        "summary": r["body"]               # 👈 the release notes
    }
    for r in releases_p1
]

pandas_releases = pd.DataFrame(records)
pandas_releases

In [ ]:
# Convert published_at from a string to a proper datetime object
# This lets us do date arithmetic (e.g. calculate time between releases)
pandas_releases['published_at'] = pd.to_datetime(pandas_releases['published_at'])
pandas_releases.dtypes

### Saving to DuckDB

Now let's persist the data to a database. We use **DuckDB** — a lightweight, file-based database that needs no server setup. `SQLAlchemy` gives pandas a way to write DataFrames directly into it using `.to_sql()`.

In [ ]:
import sqlalchemy as sqla
import os

# Get the absolute path to the project root (one level above notebooks/)
parent_dir = os.path.abspath(os.path.pardir)

# Ensure the output directory exists
os.makedirs(f'{parent_dir}/output', exist_ok=True)

# Create a connection to the DuckDB file (unit-2-4.db is created fresh for this notebook)
engine = sqla.create_engine(f'duckdb:///{parent_dir}/output/unit-2-4.db')

In [ ]:
# Write the DataFrame to a table called "pandas_releases"
# if_exists='replace' will overwrite the table if it already exists
try:
    pandas_releases.to_sql("pandas_releases", engine, if_exists='replace', index=False)
    print("✅ Saved pandas_releases to DuckDB")
except Exception as e:
    print(f"❌ Error: {e}")

In [ ]:
# Always dispose the engine when done — this closes the connection
# You can now open output/unit-2-4.db in DBeaver or any DuckDB client
engine.dispose()
print("Connection closed.")

---
### ✏️ Exercise 1: Who Has Contributed the Most Commits to pandas?

The `stats/contributors` endpoint returns a list of contributors and their commit counts.

**Your task:**
1. Call `https://api.github.com/repos/pandas-dev/pandas/stats/contributors` with the same headers as before
2. Extract the `login` (contributor username) and `total` (commit count) from each item
3. Build a DataFrame named `contributors` with columns `login` and `total_commits`
4. Find the contributor with the highest number of commits
5. Save the DataFrame to DuckDB as a table named `pandas_contributors`

> ⚠️ **Note on this endpoint:** GitHub computes contributor stats on demand. If you get a `202` status code (instead of `200`), the data isn't ready yet — wait 10 seconds and try again.

---

**💡 Hint 1 — The data structure:**  
Each item in the response list looks like this:
```python
{
    "total": 1234,         # total commits by this contributor
    "author": {
        "login": "jreback" # their GitHub username
    },
    "weeks": [...]         # weekly breakdown — ignore this
}
```

**💡 Hint 2 — Accessing nested values:**  
To get the login, use `item["author"]["login"]`. To get total commits, use `item["total"]`.

**💡 Hint 3 — Finding the top contributor:**  
After building the DataFrame, use `.sort_values("total_commits", ascending=False).head(1)` to find the contributor with the most commits.

**💡 Hint 4 — Saving to DuckDB:**  
The engine was closed after the releases save. Recreate it before saving:
```python
engine2 = sqla.create_engine(f'duckdb:///{parent_dir}/output/unit-2-4.db')
contributors.to_sql("pandas_contributors", engine2, if_exists='replace', index=False)
engine2.dispose()
```

In [ ]:
# Your code here